<a href="https://colab.research.google.com/github/danielhacobian/probing-VLMs/blob/initial-release/notebooks/pusht_layerwise_motion_probe_walkthrough_standalone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Where does PushT physics become readable?

This notebook follows the UMaze layerwise motion-probe walkthrough, but applies it to PushT. It asks which physical variables can be recovered with a ridge-linear map, from which internal tensor, and at which model layer. It uses all 18,500 trajectories in the public PushT dataset.

PushT contains two moving bodies. We therefore probe the pusher's position, velocity, acceleration, speed, and heading; the T-block's position, translational velocity, translational acceleration, speed, and heading; and the T-block's circular orientation, angular velocity, and angular acceleration.

The representation families stay separate:

- **DINO is a per-frame visual encoder.** We probe CLS, mean patch tokens, and the trained projected aggregate at each of its 12 blocks. Motion primarily uses first or second temporal differences.
- **The predictor is temporally contextual.** We probe pooled visual-token channels at each of its six predictor blocks. Predictor results have their own figures and never appear as extra DINO layers.

The checkpoints stay frozen. Only the linear readout is fitted. Readability does not establish causal use by the planner.

## One-click standalone Colab setup

This run selects one deterministic four-frame window from each of all 18,500 public PushT trajectories. Adjacent cached frames are five simulator steps apart, matching PushT training. Straightening OFF and ON use exactly the same videos, window identifiers, states, actions, labels, splits, probes, controls, and metrics.

The default path downloads the complete frozen OFF/ON activation caches from the `probing-VLMs` `pusht-probe-cache-v1` GitHub Release and refits every probe locally. This mirrors the cache-first UMaze walkthrough: collaborators can reproduce the splits, controls, tables, and graphs using only repository-owned assets. The public OSF trajectories and verified checkpoints are needed only if `PUSHT_FORCE_RECOMPUTE=1` is set to extract activations again.

Use an A100 high-memory runtime. Activations and results are written under `/content/pusht_layerwise_probe_runs/all_18500_trajectories`. The final cell packages the graphs and tables and downloads the archive to your computer. Colab runtime files are temporary, so download the archive before disconnecting.


In [ ]:
import hashlib, os, shutil, subprocess, sys, urllib.request, zipfile
from pathlib import Path

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/danielhacobian/probing-VLMs.git"
REPO_BRANCH = "initial-release"
COLAB_REPO = Path("/content/probing-VLMs")
TRAJECTORY_LIMIT = 18500
RUN_LABEL = "all_18500_trajectories"


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


if IN_COLAB:
    if not (COLAB_REPO / ".git").exists():
        subprocess.run([
            "git", "clone", "--branch", REPO_BRANCH, "--single-branch",
            REPO_URL, str(COLAB_REPO),
        ], check=True)
    else:
        subprocess.run(["git", "-C", str(COLAB_REPO), "fetch", "origin", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", str(COLAB_REPO), "checkout", "--detach", "FETCH_HEAD"], check=True)
    os.chdir(COLAB_REPO)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "decord", "einops", "omegaconf", "hydra-core==1.3.2",
    ], check=True)

REPOSITORY_CHECKPOINT_ROOT = Path("artifacts/checkpoints/pusht_paper_protocol_on_off")
CHECKPOINT_ROOT = Path(os.environ.get("PUSHT_CHECKPOINT_ROOT", REPOSITORY_CHECKPOINT_ROOT)).expanduser()
EXPECTED_CHECKPOINT_SHA256 = {
    "off": "a03cd7e514223db0f3543ce00748036f80df9397dde560684555801e41c936a5",
    "on": "de31f8345d5274cb0dbd68bdaa38e8bab601eb52c4c0f7f83ea0d85a8c20af4c",
}

FORCE_RECOMPUTE = os.environ.get("PUSHT_FORCE_RECOMPUTE", "0") == "1"
OUTPUT_DIR = Path("/content/pusht_layerwise_probe_runs") / RUN_LABEL if IN_COLAB else Path("pusht_layerwise_probe_runs") / RUN_LABEL
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print({
    "in_colab": IN_COLAB,
    "repo": str(Path.cwd()),
    "trajectory_limit": TRAJECTORY_LIMIT,
    "checkpoint_root": str(CHECKPOINT_ROOT),
    "output": str(OUTPUT_DIR),
})


## Experimental design

For a sampled representation $h_t^\ell$, DINO position and orientation probes use $h_t^\ell$, velocity probes use $\Delta h_t^\ell$, and acceleration probes use $\Delta^2h_t^\ell$. Predictor probes use the contextual feature at its matching history slot.

Every probe standardizes features with training rows only and fits ridge regression with $\lambda=10$. The controls match the UMaze notebook: episode-validation evaluation, a buffered spatial-validation split, 20 window-shuffled null fits, a pose-only baseline, pose-residualized targets, and 95% percentile intervals from resampling complete validation trajectory windows.

The environment-specific changes are limited to PushT's state and action structure. We probe the pusher and T-block separately; use the block's starting Y position for the spatial holdout; derive block translation and rotation from matched sampled states; and feed five normalized 2-D actions per model slot, exactly as PushT training does.

Block orientation is encoded as $(\cos\theta,\sin\theta)$. Raw angle regression would create a false discontinuity at $0=2\pi$.

In [ ]:
import csv, gc, json, math, random, time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from decord import VideoReader
from datasets.img_transforms import default_transform
from datasets.pusht_dset import PushTDataset

exec(compile('#!/usr/bin/env python3\n"""NumPy helpers for the PushT layerwise motion-probe Colab notebooks.\n\nThe expensive activation extraction lives in the generated notebooks because\nit must run against the exact checkpoint objects loaded in Colab.  This module\nkeeps target construction, feature alignment, leakage-resistant splits, ridge\ncontrols, uncertainty estimates, and cache I/O testable without a GPU.\n"""\n\nfrom __future__ import annotations\n\nimport json\nimport math\nimport re\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport numpy as np\n\n\ndef r2_score(y, prediction):\n    y = np.asarray(y, dtype=np.float64)\n    prediction = np.asarray(prediction, dtype=np.float64)\n    denominator = np.square(y - y.mean(axis=0)).sum()\n    return float(1.0 - np.square(y - prediction).sum() / max(denominator, 1e-12))\n\n\ndef regression_scores(truth, prediction):\n    truth = np.asarray(truth, dtype=np.float64)\n    prediction = np.asarray(prediction, dtype=np.float64)\n    return {\n        "r2": r2_score(truth, prediction),\n        "rmse": float(np.sqrt(np.mean(np.square(truth - prediction)))),\n        "mae": float(np.mean(np.abs(truth - prediction))),\n    }\n\n\ndef direction_scores(truth, prediction):\n    truth = np.asarray(truth, dtype=np.float64)\n    prediction = np.asarray(prediction, dtype=np.float64)\n    mask = np.isfinite(truth).all(axis=-1) & np.isfinite(prediction).all(axis=-1)\n    truth, prediction = truth[mask], prediction[mask]\n    cosine = np.sum(truth * prediction, axis=-1) / (\n        np.linalg.norm(truth, axis=-1) * np.linalg.norm(prediction, axis=-1) + 1e-8\n    )\n    true_theta = np.arctan2(truth[:, 1], truth[:, 0])\n    pred_theta = np.arctan2(prediction[:, 1], prediction[:, 0])\n    delta = np.arctan2(np.sin(pred_theta - true_theta), np.cos(pred_theta - true_theta))\n    return {\n        "cosine": float(np.mean(cosine)),\n        "angular_mae_deg": float(np.degrees(np.mean(np.abs(delta)))),\n    }\n\n\ndef _frame_from_first_difference(values, frame_count):\n    values = np.asarray(values, dtype=np.float64)\n    result = np.full((values.shape[0], frame_count) + values.shape[2:], np.nan)\n    result[:, 1:] = values\n    return result\n\n\ndef _frame_from_second_difference(values, frame_count):\n    values = np.asarray(values, dtype=np.float64)\n    result = np.full((values.shape[0], frame_count) + values.shape[2:], np.nan)\n    result[:, 2:] = values\n    return result\n\n\ndef _polar(vector):\n    vector = np.asarray(vector, dtype=np.float64)\n    magnitude = np.linalg.norm(vector, axis=-1)\n    direction = vector / np.maximum(magnitude[..., None], 1e-8)\n    return magnitude, direction\n\n\ndef build_pusht_targets(states: np.ndarray, frameskip: int, step_dt: float = 1.0):\n    """Build pusher, T-block, and rotational targets from PushT states.\n\n    PushT states are ``[agent_x, agent_y, block_x, block_y, block_angle,\n    agent_vx, agent_vy]``.  The block\'s translational and angular velocities\n    are derived from the same sampled frames used by the visual probes.\n    """\n    states = np.asarray(states, dtype=np.float64)\n    if states.ndim != 3 or states.shape[-1] < 5:\n        raise ValueError("PushT states must have shape [window, time, >=5]")\n    dt = float(frameskip) * float(step_dt)\n    if dt <= 0:\n        raise ValueError("frameskip * step_dt must be positive")\n\n    n, t = states.shape[:2]\n    agent_position = states[..., 0:2]\n    block_position = states[..., 2:4]\n    block_angle = np.unwrap(states[..., 4], axis=1)\n    block_orientation = np.stack([np.cos(block_angle), np.sin(block_angle)], axis=-1)\n\n    transition_agent_velocity = np.diff(agent_position, axis=1) / dt\n    if states.shape[-1] >= 7:\n        agent_velocity = states[..., 5:7]\n    else:\n        agent_velocity = _frame_from_first_difference(transition_agent_velocity, t)\n    transition_agent_acceleration = np.diff(transition_agent_velocity, axis=1) / dt\n    agent_acceleration = _frame_from_first_difference(\n        np.diff(agent_velocity, axis=1) / dt, t\n    )\n\n    transition_block_velocity = np.diff(block_position, axis=1) / dt\n    block_velocity = _frame_from_first_difference(transition_block_velocity, t)\n    transition_block_acceleration = np.diff(transition_block_velocity, axis=1) / dt\n    block_acceleration = _frame_from_second_difference(transition_block_acceleration, t)\n\n    transition_block_angular_velocity = np.diff(block_angle, axis=1) / dt\n    block_angular_velocity = _frame_from_first_difference(\n        transition_block_angular_velocity[..., None], t\n    )\n    transition_block_angular_acceleration = (\n        np.diff(transition_block_angular_velocity, axis=1) / dt\n    )\n    block_angular_acceleration = _frame_from_second_difference(\n        transition_block_angular_acceleration[..., None], t\n    )\n\n    agent_speed, agent_heading = _polar(agent_velocity)\n    agent_acceleration_magnitude, agent_acceleration_direction = _polar(\n        agent_acceleration\n    )\n    transition_agent_speed, transition_agent_heading = _polar(\n        transition_agent_velocity\n    )\n    transition_agent_acceleration_magnitude, transition_agent_acceleration_direction = _polar(\n        transition_agent_acceleration\n    )\n\n    block_speed, block_heading = _polar(block_velocity)\n    block_acceleration_magnitude, block_acceleration_direction = _polar(\n        block_acceleration\n    )\n    transition_block_speed, transition_block_heading = _polar(\n        transition_block_velocity\n    )\n    transition_block_acceleration_magnitude, transition_block_acceleration_direction = _polar(\n        transition_block_acceleration\n    )\n\n    block_pose = np.concatenate([block_position, block_orientation], axis=-1)\n    contexts = {}\n    agent_variables = (\n        "agent_position", "agent_velocity", "agent_acceleration", "agent_speed",\n        "agent_heading", "agent_acceleration_magnitude", "agent_acceleration_direction",\n    )\n    block_variables = (\n        "block_position", "block_orientation", "block_velocity", "block_acceleration",\n        "block_speed", "block_heading", "block_acceleration_magnitude",\n        "block_acceleration_direction", "block_angular_velocity",\n        "block_angular_acceleration",\n    )\n    for variable in agent_variables:\n        contexts[variable] = agent_position\n    for variable in block_variables:\n        contexts[variable] = block_pose\n\n    return {\n        "agent_position": agent_position,\n        "agent_velocity": agent_velocity,\n        "agent_acceleration": agent_acceleration,\n        "agent_speed": agent_speed,\n        "agent_heading": agent_heading,\n        "agent_acceleration_magnitude": agent_acceleration_magnitude,\n        "agent_acceleration_direction": agent_acceleration_direction,\n        "transition_agent_velocity": transition_agent_velocity,\n        "transition_agent_acceleration": transition_agent_acceleration,\n        "transition_agent_speed": transition_agent_speed,\n        "transition_agent_heading": transition_agent_heading,\n        "transition_agent_acceleration_magnitude": transition_agent_acceleration_magnitude,\n        "transition_agent_acceleration_direction": transition_agent_acceleration_direction,\n        "block_position": block_position,\n        "block_orientation": block_orientation,\n        "block_velocity": block_velocity,\n        "block_acceleration": block_acceleration,\n        "block_speed": block_speed,\n        "block_heading": block_heading,\n        "block_acceleration_magnitude": block_acceleration_magnitude,\n        "block_acceleration_direction": block_acceleration_direction,\n        "transition_block_velocity": transition_block_velocity,\n        "transition_block_acceleration": transition_block_acceleration,\n        "transition_block_speed": transition_block_speed,\n        "transition_block_heading": transition_block_heading,\n        "transition_block_acceleration_magnitude": transition_block_acceleration_magnitude,\n        "transition_block_acceleration_direction": transition_block_acceleration_direction,\n        "block_angular_velocity": block_angular_velocity,\n        "block_angular_acceleration": block_angular_acceleration,\n        "transition_block_angular_velocity": transition_block_angular_velocity[..., None],\n        "transition_block_angular_acceleration": transition_block_angular_acceleration[..., None],\n        "contexts": contexts,\n        "dt": dt,\n        "num_windows": n,\n    }\n\n\nFIRST_DIFFERENCE_TARGETS = {\n    "agent_velocity": "transition_agent_velocity",\n    "agent_speed": "transition_agent_speed",\n    "agent_heading": "transition_agent_heading",\n    "block_velocity": "transition_block_velocity",\n    "block_speed": "transition_block_speed",\n    "block_heading": "transition_block_heading",\n    "block_angular_velocity": "transition_block_angular_velocity",\n}\n\nSECOND_DIFFERENCE_TARGETS = {\n    "agent_acceleration": "transition_agent_acceleration",\n    "agent_acceleration_magnitude": "transition_agent_acceleration_magnitude",\n    "agent_acceleration_direction": "transition_agent_acceleration_direction",\n    "block_acceleration": "transition_block_acceleration",\n    "block_acceleration_magnitude": "transition_block_acceleration_magnitude",\n    "block_acceleration_direction": "transition_block_acceleration_direction",\n    "block_angular_acceleration": "transition_block_angular_acceleration",\n}\n\n\ndef align_representation(rep: np.ndarray, targets: dict, variable: str, mode: str):\n    """Return matched features, labels, and pose-only control features."""\n    rep = np.asarray(rep)\n    if rep.ndim == 4:\n        rep = rep.mean(axis=2)\n    if rep.ndim != 3:\n        raise ValueError("representation must have shape [window, time, feature]")\n    t = rep.shape[1]\n    if variable not in targets["contexts"]:\n        raise KeyError(f"unknown PushT variable {variable!r}")\n    context = targets["contexts"][variable][:, :t]\n\n    if mode == "frame":\n        return rep, targets[variable][:, :t], context\n    if mode in ("delta", "concat"):\n        if variable not in FIRST_DIFFERENCE_TARGETS:\n            raise ValueError(f"{variable!r} has no first-difference target")\n        if t < 2:\n            raise ValueError("at least two representation slots are required")\n        features = np.diff(rep, axis=1) if mode == "delta" else np.concatenate(\n            [rep[:, :-1], rep[:, 1:]], axis=-1\n        )\n        labels = targets[FIRST_DIFFERENCE_TARGETS[variable]][:, : t - 1]\n        pose_context = 0.5 * (context[:, :-1] + context[:, 1:])\n        return features, labels, pose_context\n    if mode in ("second_delta", "concat3"):\n        if variable not in SECOND_DIFFERENCE_TARGETS:\n            raise ValueError(f"{variable!r} has no second-difference target")\n        if t < 3:\n            raise ValueError("at least three representation slots are required")\n        features = (\n            rep[:, 2:] - 2.0 * rep[:, 1:-1] + rep[:, :-2]\n            if mode == "second_delta"\n            else np.concatenate([rep[:, :-2], rep[:, 1:-1], rep[:, 2:]], axis=-1)\n        )\n        labels = targets[SECOND_DIFFERENCE_TARGETS[variable]][:, : t - 2]\n        return features, labels, context[:, 1:-1]\n    raise ValueError(f"unknown mode {mode!r}")\n\n\ndef episode_group_split(choices, test_fraction: float = 0.2, seed: int = 0):\n    choices = np.asarray(choices, dtype=np.int64)\n    episodes = np.unique(choices[:, 0])\n    if len(episodes) < 2:\n        raise ValueError("episode-held-out evaluation requires at least two episodes")\n    rng = np.random.default_rng(seed)\n    shuffled = rng.permutation(episodes)\n    n_test = min(len(episodes) - 1, max(1, int(math.ceil(test_fraction * len(episodes)))))\n    test_episodes = set(shuffled[:n_test].tolist())\n    test = np.asarray([i for i, episode in enumerate(choices[:, 0]) if episode in test_episodes])\n    train = np.asarray([i for i, episode in enumerate(choices[:, 0]) if episode not in test_episodes])\n    return train, test\n\n\ndef spatial_holdout_split(\n    anchor_position: np.ndarray,\n    axis: int = 1,\n    quantile: float = 0.8,\n    high: bool = True,\n    buffer_fraction: float = 0.05,\n):\n    anchor_position = np.asarray(anchor_position)\n    coordinate = anchor_position[:, axis]\n    boundary = float(np.quantile(coordinate, quantile if high else 1.0 - quantile))\n    buffer = float(np.ptp(coordinate) * buffer_fraction)\n    if high:\n        train = np.flatnonzero(coordinate < boundary - buffer)\n        test = np.flatnonzero(coordinate >= boundary)\n    else:\n        train = np.flatnonzero(coordinate > boundary + buffer)\n        test = np.flatnonzero(coordinate <= boundary)\n    if not len(train) or not len(test):\n        raise ValueError("spatial split produced an empty train or test set")\n    return train, test, {\n        "boundary": boundary,\n        "buffer": buffer,\n        "axis": axis,\n        "high": high,\n        "anchor": "block_position",\n    }\n\n\ndef _row_finite(values):\n    values = np.asarray(values)\n    return np.isfinite(values).all(axis=-1) if values.ndim > 1 else np.isfinite(values)\n\n\ndef _flatten_windows(values, window_indices):\n    values = np.asarray(values)[window_indices]\n    return values.reshape(-1, *values.shape[2:])\n\n\n@dataclass\nclass RidgeDesign:\n    x_mean: np.ndarray\n    x_std: np.ndarray\n    xs_train: np.ndarray\n    xs_test: np.ndarray\n    gram: np.ndarray\n    train_valid: np.ndarray\n    test_valid: np.ndarray\n\n    @classmethod\n    def prepare(cls, features, labels, train_idx, test_idx, ridge):\n        x_train = _flatten_windows(features, train_idx).astype(np.float64, copy=False)\n        x_test = _flatten_windows(features, test_idx).astype(np.float64, copy=False)\n        y_train = _flatten_windows(labels, train_idx)\n        y_test = _flatten_windows(labels, test_idx)\n        train_valid = np.isfinite(x_train).all(axis=-1) & _row_finite(y_train)\n        test_valid = np.isfinite(x_test).all(axis=-1) & _row_finite(y_test)\n        x_train = x_train[train_valid]\n        x_test = x_test[test_valid]\n        x_mean = x_train.mean(0, keepdims=True)\n        x_std = x_train.std(0, keepdims=True)\n        x_std[x_std < 1e-6] = 1.0\n        xs_train = (x_train - x_mean) / x_std\n        xs_test = (x_test - x_mean) / x_std\n        gram = xs_train.T @ xs_train + float(ridge) * np.eye(xs_train.shape[1])\n        return cls(x_mean, x_std, xs_train, xs_test, gram, train_valid, test_valid)\n\n    def select_labels(self, labels, train_idx, test_idx):\n        y_train = _flatten_windows(labels, train_idx)[self.train_valid]\n        y_test = _flatten_windows(labels, test_idx)[self.test_valid]\n        return y_train, y_test\n\n    def fit_arrays(self, y_train, y_test):\n        y_train = np.asarray(y_train, dtype=np.float64)\n        y_test = np.asarray(y_test, dtype=np.float64)\n        scalar = y_train.ndim == 1\n        if scalar:\n            y_train = y_train[:, None]\n        y_mean = y_train.mean(0, keepdims=True)\n        centered = y_train - y_mean\n        weight = np.linalg.solve(self.gram, self.xs_train.T @ centered)\n        prediction = self.xs_test @ weight + y_mean\n        if scalar:\n            prediction = prediction[:, 0]\n        return y_test, prediction, weight\n\n    def fit(self, labels, train_idx, test_idx):\n        return self.fit_arrays(*self.select_labels(labels, train_idx, test_idx))\n\n\ndef _validity_key(labels, train_idx, test_idx):\n    train_valid = _row_finite(_flatten_windows(labels, train_idx))\n    test_valid = _row_finite(_flatten_windows(labels, test_idx))\n    return (np.packbits(train_valid).tobytes(), np.packbits(test_valid).tobytes())\n\n\ndef get_design(cache, cache_prefix, features, labels, train_idx, test_idx, ridge):\n    key = (cache_prefix, _validity_key(labels, train_idx, test_idx))\n    if key not in cache:\n        cache[key] = RidgeDesign.prepare(features, labels, train_idx, test_idx, ridge)\n    return cache[key]\n\n\ndef vectorized_shuffled_label_scores(\n    design: RidgeDesign,\n    labels,\n    train_idx,\n    test_idx,\n    repeats=20,\n    seed=0,\n):\n    """Refit all window-shuffled nulls with one shared ridge factorization."""\n    labels = np.asarray(labels)\n    rng = np.random.default_rng(seed)\n    original_train, original_test = design.select_labels(labels, train_idx, test_idx)\n    scalar = original_train.ndim == 1\n    if scalar:\n        original_train = original_train[:, None]\n        original_test = original_test[:, None]\n    blocks = []\n    for _ in range(repeats):\n        shuffled = labels[np.asarray(train_idx)[rng.permutation(len(train_idx))]]\n        flat = shuffled.reshape(-1, *shuffled.shape[2:])\n        candidate = flat[design.train_valid]\n        if candidate.ndim == 1:\n            candidate = candidate[:, None]\n        blocks.append(candidate)\n    stacked = np.concatenate(blocks, axis=1).astype(np.float64, copy=False)\n    y_mean = stacked.mean(0, keepdims=True)\n    weight = np.linalg.solve(design.gram, design.xs_train.T @ (stacked - y_mean))\n    prediction = design.xs_test @ weight + y_mean\n    width = original_test.shape[1]\n    scores = []\n    for index in range(repeats):\n        block = prediction[:, index * width : (index + 1) * width]\n        truth = original_test\n        if scalar:\n            block = block[:, 0]\n            truth = truth[:, 0]\n        scores.append(r2_score(truth, block))\n    return np.asarray(scores)\n\n\ndef residualize_against_context(labels, context, train_idx, ridge=10.0):\n    labels = np.asarray(labels, dtype=np.float64)\n    context = np.asarray(context, dtype=np.float64)\n    all_indices = np.arange(len(labels))\n    design = RidgeDesign.prepare(context, labels, train_idx, all_indices, ridge)\n    y_train, _ = design.select_labels(labels, train_idx, all_indices)\n    all_labels = _flatten_windows(labels, all_indices)[design.test_valid]\n    _, prediction, _ = design.fit_arrays(y_train, all_labels)\n    flat = labels.reshape(-1, *labels.shape[2:])\n    residual = np.full_like(flat, np.nan, dtype=np.float64)\n    residual[design.test_valid] = all_labels - prediction\n    return residual.reshape(labels.shape)\n\n\ndef bootstrap_metric_ci(truth, prediction, metric="r2", repeats=1000, seed=0):\n    truth, prediction = np.asarray(truth), np.asarray(prediction)\n    rng = np.random.default_rng(seed)\n    values = []\n    for _ in range(repeats):\n        index = rng.integers(0, len(truth), len(truth))\n        if metric == "r2":\n            values.append(r2_score(truth[index], prediction[index]))\n        elif metric == "cosine":\n            values.append(direction_scores(truth[index], prediction[index])["cosine"])\n        else:\n            raise ValueError(f"unsupported metric {metric!r}")\n    return np.quantile(values, [0.025, 0.975]).tolist()\n\n\ndef mask_slow_directions(labels, magnitude, train_idx, quantile=0.1):\n    labels = np.asarray(labels, dtype=float).copy()\n    magnitude = np.asarray(magnitude)\n    cutoff = float(np.nanquantile(magnitude[train_idx], quantile))\n    labels[magnitude <= max(cutoff, 1e-8)] = np.nan\n    return labels, cutoff\n\n\ndef readability_onset(rows, value_key, control_key, consecutive=2, fraction_of_peak=0.5):\n    ordered = sorted(rows, key=lambda row: int(row["layer"]))\n    peak = max(float(row[value_key]) for row in ordered)\n    qualifies = [\n        float(row[value_key]) > max(float(row[control_key]), 0.0)\n        and float(row[value_key]) >= fraction_of_peak * peak\n        for row in ordered\n    ]\n    for index in range(0, len(ordered) - consecutive + 1):\n        if all(qualifies[index : index + consecutive]):\n            return int(ordered[index]["layer"])\n    return None\n\n\ndef _safe_name(name):\n    return re.sub(r"[^A-Za-z0-9_.-]+", "__", name)\n\n\ndef save_activation_cache(path, representations, states, actions, choices, metadata=None):\n    """Save each representation separately for bounded-memory Colab reloads."""\n    path = Path(path)\n    rep_dir = path / "representations"\n    rep_dir.mkdir(parents=True, exist_ok=True)\n    manifest = {}\n    for name in sorted(representations):\n        filename = f"{_safe_name(name)}.npy"\n        array = np.asarray(representations[name], dtype=np.float16)\n        np.save(rep_dir / filename, array, allow_pickle=False)\n        manifest[name] = {"file": filename, "shape": list(array.shape), "dtype": str(array.dtype)}\n    np.save(path / "states.npy", np.asarray(states, dtype=np.float32), allow_pickle=False)\n    np.save(path / "actions.npy", np.asarray(actions, dtype=np.float32), allow_pickle=False)\n    np.save(path / "choices.npy", np.asarray(choices, dtype=np.int64), allow_pickle=False)\n    (path / "manifest.json").write_text(json.dumps({\n        "metadata": metadata or {}, "representations": manifest\n    }, indent=2))\n\n\ndef load_activation_cache(path, mmap_mode="r"):\n    path = Path(path)\n    payload = json.loads((path / "manifest.json").read_text())\n    representations = {\n        name: np.load(path / "representations" / item["file"], mmap_mode=mmap_mode, allow_pickle=False)\n        for name, item in payload["representations"].items()\n    }\n    return (\n        representations,\n        np.load(path / "states.npy", mmap_mode=mmap_mode, allow_pickle=False),\n        np.load(path / "actions.npy", mmap_mode=mmap_mode, allow_pickle=False),\n        np.load(path / "choices.npy", mmap_mode=mmap_mode, allow_pickle=False),\n        payload["metadata"],\n    )\n\n', "pusht_probe_walkthrough_embedded.py", "exec"), globals())
# The 18,500-window cache is stored as float16. Cast representations before
# taking temporal differences; otherwise small motion signals are differenced
# in half precision. The underlying helper and ridge fit remain unchanged.
_align_representation_base = align_representation
def align_representation(rep, targets, variable, mode):
    return _align_representation_base(np.asarray(rep, dtype=np.float32), targets, variable, mode)

from scripts.umaze_probe_walkthrough import (
    episode_group_train_val_test_split, fit_probe_grouped,
    group_flat_predictions_by_window, grouped_metric,
    grouped_regression_summary, select_then_test_representations,
    trajectory_bootstrap_metric_ci,
)

plt.style.use("seaborn-v0_8-whitegrid")

SEED = 0
RIDGE = 10.0
NUM_FRAMES = 4
FRAME_SKIP = 5
STEP_DT = 1.0
BATCH_SIZE = 16
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
BOOTSTRAP_REPEATS = 1000
SHUFFLE_REPEATS = 20

CHECKPOINT_SHA256 = {
    "off": "a03cd7e514223db0f3543ce00748036f80df9397dde560684555801e41c936a5",
    "on": "de31f8345d5274cb0dbd68bdaa38e8bab601eb52c4c0f7f83ea0d85a8c20af4c",
}
CHECKPOINTS = {
    condition: CHECKPOINT_ROOT / condition / "model_latest.pth"
    for condition in ("off", "on")
}
CHECKPOINTS_AVAILABLE = all(
    path.is_file() and sha256(path) == CHECKPOINT_SHA256[condition]
    for condition, path in CHECKPOINTS.items()
)
if FORCE_RECOMPUTE and not CHECKPOINTS_AVAILABLE:
    raise FileNotFoundError(
        "PUSHT_FORCE_RECOMPUTE=1 requires verified off/model_latest.pth and "
        "on/model_latest.pth files under PUSHT_CHECKPOINT_ROOT."
    )

CONFIGS = {
    condition: CHECKPOINT_ROOT / condition / "hydra.yaml"
    for condition in ("off", "on")
}
for path in CONFIGS.values():
    assert path.exists(), path

def write_rows(path, rows):
    rows = list(rows)
    if not rows:
        return
    fieldnames = list(dict.fromkeys(key for row in rows for key in row))
    with Path(path).open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader(); writer.writerows(rows)

def show_rows(rows, columns=None, limit=16):
    rows = list(rows)[:limit]
    if not rows:
        print("(no rows)"); return
    columns = columns or list(rows[0])
    widths = {
        key: min(38, max(len(key), *(len(f"{row.get(key, '')}") for row in rows)))
        for key in columns
    }
    print(" | ".join(key.ljust(widths[key]) for key in columns))
    print("-+-".join("-" * widths[key] for key in columns))
    for row in rows:
        print(" | ".join(f"{row.get(key, '')}"[:widths[key]].ljust(widths[key]) for key in columns))

def grouped(rows, keys):
    result = {}
    for row in rows:
        key = tuple(row[name] for name in keys)
        result.setdefault(key, []).append(row)
    return result

class CombinedPushTDataset:
    def __init__(self, parts):
        self.parts = list(parts)
        self.offsets = np.cumsum([0] + [len(part) for part in self.parts])
        self.seq_lengths = np.concatenate([
            np.asarray(part.seq_lengths, dtype=np.int64) for part in self.parts
        ])

    def __len__(self):
        return int(self.offsets[-1])

    def _locate(self, index):
        part_index = int(np.searchsorted(self.offsets[1:], index, side="right"))
        return self.parts[part_index], int(index - self.offsets[part_index])

    def get_frames(self, index, frames):
        part, local = self._locate(index)
        return part.get_frames(local, frames)

    def action_slice(self, index, start, stop):
        part, local = self._locate(index)
        return part.actions[local, start:stop]

def find_dataset_parts(root):
    candidates = []
    for states_file in Path(root).rglob("states.pth"):
        folder = states_file.parent
        if (folder / "seq_lengths.pkl").exists() and (folder / "obses").is_dir():
            candidates.append(folder)
    return sorted(set(candidates))

def load_checkpoint_modules(path, device):
    from models.dino import DinoV2Encoder
    _ = DinoV2Encoder("dinov2_vits14", "x_norm_patchtokens")
    try:
        payload = torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        payload = torch.load(path, map_location="cpu")
    modules = {}
    for name in ("encoder", "predictor", "proprio_encoder", "action_encoder"):
        if name not in payload:
            raise KeyError(f"checkpoint is missing {name!r}")
        modules[name] = payload[name].to(device).eval()
    print({
        "checkpoint": str(path), "epoch": payload.get("epoch"),
        "dino_blocks": len(modules["encoder"].base_model.blocks),
        "predictor_blocks": len(modules["predictor"].transformer.layers),
    })
    return modules, int(payload.get("epoch", -1))

def load_batch(dataset, batch_choices):
    visuals, proprios, actions, states = [], [], [], []
    for episode, start in batch_choices:
        indices = [int(start) + FRAME_SKIP * offset for offset in range(NUM_FRAMES)]
        obs, _, state, _ = dataset.get_frames(int(episode), indices)
        # Match TrajSlicerDataset exactly: each sampled model slot receives
        # the five normalized 2-D actions beginning at that slot (10 channels).
        # The predictor consumes the first three history slots; the fourth
        # action group is kept because training constructed four model slots.
        action = dataset.action_slice(
            int(episode), int(start), int(start) + FRAME_SKIP * NUM_FRAMES
        )
        if action.shape[0] != FRAME_SKIP * NUM_FRAMES:
            raise ValueError(f"incomplete action window for {episode=} {start=}")
        action = action.reshape(NUM_FRAMES, FRAME_SKIP * action.shape[-1])
        if obs["visual"].shape[0] != NUM_FRAMES or obs["proprio"].shape[0] != NUM_FRAMES:
            raise ValueError(f"incomplete observation window for {episode=} {start=}")
        visuals.append(obs["visual"]); proprios.append(obs["proprio"])
        actions.append(action); states.append(state)
    return tuple(torch.stack(items) for items in (visuals, proprios, actions, states))

def encode_stream(module, values, name):
    expected = int(module.patch_embed.in_channels)
    actual = int(values.shape[-1])
    if expected != actual:
        raise ValueError(f"{name} has {actual} channels; checkpoint expects {expected}")
    return module(values)

def collect_activations(modules, dataset, choices):
    encoder = modules["encoder"]
    predictor = modules["predictor"]
    representations = {}
    all_states, all_actions = [], []
    visual_dim = int(encoder.emb_dim)
    if hasattr(encoder, "agg_mlp"):
        token_count = int(encoder.agg_mlp[0].in_features // visual_dim)
        token_side = math.isqrt(token_count)
        if token_side * token_side != token_count:
            raise ValueError(f"checkpoint aggregate expects non-square token count {token_count}")
        encoder_input_size = token_side * int(encoder.patch_size)
    else:
        encoder_input_size = 224
    started = time.time()

    def append(key, value):
        representations.setdefault(key, []).append(value.detach().half().cpu())

    with torch.inference_mode():
        for batch_start in range(0, len(choices), BATCH_SIZE):
            batch_choices = choices[batch_start:batch_start + BATCH_SIZE]
            visual, proprio, action, state = load_batch(dataset, batch_choices)
            b, t = visual.shape[:2]
            flat = visual.to(DEVICE).reshape(b * t, *visual.shape[2:])
            if flat.shape[-2:] != (encoder_input_size, encoder_input_size):
                flat = F.interpolate(
                    flat, size=(encoder_input_size, encoder_input_size),
                    mode="bilinear", align_corners=False,
                )
            layer_outputs = encoder.forward_intermediates(flat)
            for output in layer_outputs:
                layer = output["layer"]
                append(f"dino/{layer}/cls", output["cls"].reshape(b, t, -1))
                append(
                    f"dino/{layer}/pooled_patches",
                    output["pooled_patches"].reshape(b, t, -1),
                )
                if "aggregated" in output:
                    append(
                        f"dino/{layer}/projected_aggregate",
                        output["aggregated"].reshape(b, t, -1),
                    )

            visual_tokens = encoder(flat).reshape(b, t, -1, visual_dim)
            prop_emb = encode_stream(
                modules["proprio_encoder"], proprio.to(DEVICE), "proprio"
            )
            act_emb = encode_stream(
                modules["action_encoder"], action.to(DEVICE), "action"
            )
            prop_tiled = prop_emb.unsqueeze(2).expand(-1, -1, visual_tokens.shape[2], -1)
            act_tiled = act_emb.unsqueeze(2).expand(-1, -1, visual_tokens.shape[2], -1)
            z = torch.cat([visual_tokens, prop_tiled, act_tiled], dim=-1)
            hist = int(predictor.pos_embedding.shape[1] // z.shape[2])
            if hist != NUM_FRAMES - 1:
                raise ValueError(
                    f"checkpoint expects {hist} predictor history slots; notebook expects {NUM_FRAMES - 1}"
                )
            pred_input = z[:, :hist].reshape(b, hist * z.shape[2], -1)
            _, pred_layers = predictor(pred_input, return_intermediates=True)
            for layer, activation in enumerate(pred_layers):
                activation = activation.reshape(b, hist, z.shape[2], -1)[..., :visual_dim]
                append(f"predictor/{layer}/pooled_visual", activation.mean(dim=2))

            all_states.append(state.float().cpu())
            all_actions.append(action.float().cpu())
            done = min(batch_start + BATCH_SIZE, len(choices))
            if done == len(choices) or done % (BATCH_SIZE * 25) == 0:
                print(f"activation windows {done}/{len(choices)} in {(time.time()-started)/60:.1f} min")

    arrays = {key: torch.cat(value).numpy() for key, value in representations.items()}
    return arrays, torch.cat(all_states).numpy(), torch.cat(all_actions).numpy()

print({
    "device": str(DEVICE), "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "ridge": RIDGE, "bootstrap": BOOTSTRAP_REPEATS, "shuffles": SHUFFLE_REPEATS,
    "checkpoint_shas": CHECKPOINT_SHA256, "checkpoints_available": CHECKPOINTS_AVAILABLE,
})

## 1. Collect intermediate activations from 18,500 trajectories

The current public PushT archive exposes 18,706 trajectories across its train and validation folders. The paper protocol names 18,500 trajectories, so the full notebook selects exactly 18,500 unique trajectories with the fixed seed. This notebook selects one production-valid four-frame window from every chosen trajectory. As in training, each frame slot is paired with five consecutive normalized 2-D actions, giving a 10-channel action vector. The predictor uses the first three slots as history and the fourth frame as the prediction target.

The cache contains DINO CLS, pooled patches, and projected aggregate tensors for blocks 0 through 11. Predictor pooled visual tensors for blocks 0 through 5 live under their own family names and plots.

In [ ]:
CACHE_RELEASE_BASE = "https://github.com/danielhacobian/probing-VLMs/releases/download/pusht-probe-cache-v1"
CACHE_ASSETS = {
    "off": {"name": "pusht_activation_cache_off_18500.tar", "sha256": "b73a4f0e73a8746b5121c774d3574ab6feafff1b3512f9ee8f2f761c811ddd77"},
    "on": {"name": "pusht_activation_cache_on_18500.tar", "sha256": "f99359a21873d009cec5aefc0d78ea88305771726d8fa5b86c4acbb4a1c6521f"},
}
CACHE_DIRS = {condition: OUTPUT_DIR / f"activation_cache_{condition}" for condition in ("off", "on")}

def download_release_cache(condition):
    destination = CACHE_DIRS[condition]
    if (destination / "manifest.json").exists():
        return
    asset = CACHE_ASSETS[condition]
    archive = OUTPUT_DIR / asset["name"]
    if not archive.is_file() or sha256(archive) != asset["sha256"]:
        temporary = archive.with_suffix(archive.suffix + ".download")
        print(f"Downloading verified {condition.upper()} cache from the GitHub Release...")
        urllib.request.urlretrieve(f"{CACHE_RELEASE_BASE}/{asset['name']}", temporary)
        if sha256(temporary) != asset["sha256"]:
            raise IOError(f"Checksum mismatch for {asset['name']}")
        temporary.replace(archive)
    if destination.exists():
        shutil.rmtree(destination)
    destination.mkdir(parents=True)
    subprocess.run(["tar", "-xf", str(archive), "--strip-components=1", "-C", str(destination)], check=True)
    if not (destination / "manifest.json").exists():
        raise FileNotFoundError(f"Downloaded {condition} cache is missing manifest.json")

if not FORCE_RECOMPUTE:
    for condition in ("off", "on"):
        download_release_cache(condition)
    off_cache = CACHE_DIRS["off"]
    representations, states, actions, choices, cache_metadata = load_activation_cache(off_cache)
    cached_choices = choices
    assert choices.shape == (TRAJECTORY_LIMIT, 2), choices.shape
    assert len(np.unique(choices[:, 0])) == TRAJECTORY_LIMIT
    assert int(cache_metadata["num_windows"]) == TRAJECTORY_LIMIT
    assert cache_metadata["checkpoint_sha256"] == CHECKPOINT_SHA256["off"]
    assert states.shape == (TRAJECTORY_LIMIT, NUM_FRAMES, 7), states.shape
    assert actions.shape == (TRAJECTORY_LIMIT, NUM_FRAMES, FRAME_SKIP * 2), actions.shape
    assert len(representations) == 42, len(representations)
else:
    DATA_URL = "https://osf.io/download/k2d8w/"
    DATA_SHA256 = "442f5dee246edf670964ed7bdecd248683cd6d00580fa0e4d458abb53f92da08"
    DATA_ARCHIVE = Path("/content/pusht_noise.zip") if IN_COLAB else Path("pusht_noise.zip")
    if not DATA_ARCHIVE.exists() or sha256(DATA_ARCHIVE) != DATA_SHA256:
        temporary = DATA_ARCHIVE.with_suffix(".zip.download")
        print("Downloading the 2.8 GB public PushT archive from OSF...")
        urllib.request.urlretrieve(DATA_URL, temporary)
        assert sha256(temporary) == DATA_SHA256
        temporary.replace(DATA_ARCHIVE)
    EXTRACT_ROOT = Path("/content/pusht_data")
    marker = EXTRACT_ROOT / ".complete"
    if not marker.exists():
        if EXTRACT_ROOT.exists(): shutil.rmtree(EXTRACT_ROOT)
        EXTRACT_ROOT.mkdir(parents=True)
        subprocess.run(["unzip", "-q", str(DATA_ARCHIVE), "-d", str(EXTRACT_ROOT)], check=True)
        marker.write_text(DATA_SHA256)
    part_dirs = find_dataset_parts(EXTRACT_ROOT)
    parts = [PushTDataset(data_path=str(path), transform=default_transform(224), normalize_action=True, with_velocity=True) for path in part_dirs]
    dataset = CombinedPushTDataset(parts)
    valid_episodes = [episode for episode, length in enumerate(dataset.seq_lengths) if int(length) - FRAME_SKIP * NUM_FRAMES >= 0]
    episode_order = list(valid_episodes); random.Random(SEED).shuffle(episode_order)
    selected_episodes = sorted(episode_order[:TRAJECTORY_LIMIT])
    choices = []
    for episode in selected_episodes:
        max_start = int(dataset.seq_lengths[episode]) - FRAME_SKIP * NUM_FRAMES
        episode_rng = random.Random(SEED * 1_000_003 + int(episode))
        choices.append((episode, episode_rng.randint(0, max_start)))
    choices = np.asarray(choices, dtype=np.int64)
    assert choices.shape == (TRAJECTORY_LIMIT, 2)
    off_cache = CACHE_DIRS["off"]
    modules, epoch = load_checkpoint_modules(CHECKPOINTS["off"], DEVICE)
    representations, states, actions = collect_activations(modules, dataset, choices)
    cache_metadata = {"condition": "off", "checkpoint_sha256": CHECKPOINT_SHA256["off"], "epoch": epoch, "straighten": False, "seed": SEED, "num_windows": len(choices), "num_frames": NUM_FRAMES, "frameskip": FRAME_SKIP, "ridge": RIDGE}
    save_activation_cache(off_cache, representations, states, actions, choices, cache_metadata)
    del modules; gc.collect(); torch.cuda.empty_cache()
inventory = [
    {"representation": name, "shape": str(value.shape), "size_mb": value.nbytes / 2**20}
    for name, value in sorted(representations.items())
]
show_rows(inventory, limit=50)
print({
    "trajectories": len(choices), "unique_episodes": len(np.unique(choices[:, 0])),
    "representations": len(representations), "cache": str(off_cache),
})

## 2. Construct physical targets and inspect shortcut risk

State channels are `[agent_x, agent_y, block_x, block_y, block_angle, agent_vx, agent_vy]`. Agent velocity uses the recorded simulator value for contextual frame probes and displacement for DINO temporal probes. The block has no stored velocity channels, so its translation and rotation derivatives come from the matched sampled states.

The overview separates pusher motion, block translation, and block rotation. This matters because a model can read the pusher cleanly while failing to represent the T-block's pose or angular dynamics.

In [ ]:
targets = build_pusht_targets(states, FRAME_SKIP, STEP_DT)
agent_position = targets["agent_position"].reshape(-1, 2)
block_position = targets["block_position"].reshape(-1, 2)
agent_speed = targets["agent_speed"].reshape(-1)
block_speed = targets["block_speed"].reshape(-1)
angular_velocity = targets["block_angular_velocity"].reshape(-1)

fig, axes = plt.subplots(2, 3, figsize=(17, 9), constrained_layout=True)
a = axes[0, 0].scatter(agent_position[:, 0], agent_position[:, 1], c=agent_speed, s=5, cmap="viridis")
axes[0, 0].set(title="Pusher speed by position", xlabel="Agent x", ylabel="Agent y", aspect="equal")
fig.colorbar(a, ax=axes[0, 0], label="Pusher speed")
b = axes[0, 1].scatter(block_position[:, 0], block_position[:, 1], c=block_speed, s=5, cmap="magma")
axes[0, 1].set(title="T-block speed by position", xlabel="Block x", ylabel="Block y", aspect="equal")
fig.colorbar(b, ax=axes[0, 1], label="Block speed")
axes[0, 2].hist(states[..., 4].reshape(-1), bins=48)
axes[0, 2].set(title="T-block orientation", xlabel="Wrapped angle in radians", ylabel="Samples")
axes[1, 0].hist(agent_speed[np.isfinite(agent_speed)], bins=48)
axes[1, 0].set(title="Pusher speed labels", xlabel="Speed", ylabel="Samples")
axes[1, 1].hist(block_speed[np.isfinite(block_speed)], bins=48)
axes[1, 1].set(title="T-block speed labels", xlabel="Speed", ylabel="Samples")
axes[1, 2].hist(angular_velocity[np.isfinite(angular_velocity)], bins=48)
axes[1, 2].set(title="T-block angular velocity", xlabel="Radians per environment step", ylabel="Samples")
fig.savefig(OUTPUT_DIR / "dataset_motion_overview.png", dpi=180)
plt.show()

## 3. Leakage-resistant evaluation splits

- **Episode validation:** complete trajectories are assigned to training, validation, or a locked test partition.
- **Spatial validation:** the highest 20 percent of development-only initial T-block Y positions is validation-only. A five-percent workspace buffer is removed from training.

Exploratory layer curves use training and validation only. The locked test trajectories are evaluated once after selecting a representation on validation data.

In [ ]:
episode_train, episode_validation, episode_test = episode_group_train_val_test_split(
    choices, validation_fraction=0.2, test_fraction=0.2, seed=SEED
)
development_idx = np.sort(np.concatenate([episode_train, episode_validation]))
anchor_position = targets["block_position"][:, 0]
spatial_train_local, spatial_validation_local, spatial_config = spatial_holdout_split(
    anchor_position[development_idx], axis=1, quantile=0.8, high=True, buffer_fraction=0.05
)
spatial_train = development_idx[spatial_train_local]
spatial_validation = development_idx[spatial_validation_local]
splits = {
    "episode_validation": (episode_train, episode_validation),
    "spatial_validation": (spatial_train, spatial_validation),
}
print({"trajectory_split": {"train": len(episode_train), "validation": len(episode_validation), "test": len(episode_test)}})
print({"development_holdouts": {name: {"fit": len(train), "held_out": len(test)} for name, (train, test) in splits.items()}})
print("Spatial split:", spatial_config)

display_rng = np.random.default_rng(SEED)
def display_subset(indices, limit):
    indices = np.asarray(indices)
    if len(indices) <= limit:
        return indices
    return np.sort(display_rng.choice(indices, size=limit, replace=False))

figure_splits = (
    ("Episode validation", episode_train, episode_validation),
    ("Spatial validation", spatial_train, spatial_validation),
)
fig, axes = plt.subplots(1, 2, figsize=(12, 5.2), constrained_layout=True)
for ax, (title, train_idx, test_idx) in zip(axes, figure_splits):
    train_display = display_subset(train_idx, 2000)
    test_display = display_subset(test_idx, 600)
    ax.scatter(
        anchor_position[train_display, 0], anchor_position[train_display, 1],
        s=10, alpha=0.72, color="#4C78A8", edgecolors="none",
        label="Probe training",
    )
    ax.scatter(
        anchor_position[test_display, 0], anchor_position[test_display, 1],
        s=10, alpha=0.82, color="#F28E2B", edgecolors="none",
        label="Held-out validation",
    )
    ax.set(
        title=title, xlabel="Initial T-block x", ylabel="Initial T-block y",
        aspect="equal",
    )
    ax.grid(alpha=0.18, linewidth=0.6)
    ax.legend(loc="lower right", fontsize=8, framealpha=0.95)
fig.suptitle("PushT evaluation splits", fontsize=14)
fig.savefig(OUTPUT_DIR / "spatial_split.png", dpi=180, bbox_inches="tight")
plt.show()

## 4. Fit every layer with controls

Each score fits a fresh ridge readout for one representation, target, feature construction, and split. The cached model activations never receive gradients.

The pose-only baseline uses pusher XY for pusher targets and T-block XY plus circular orientation for block targets. The residual score removes the component linearly predictable from that pose before fitting the representation probe.

In [ ]:
CARTESIAN_SPECS_DINO = [
    ("agent_position", "frame"), ("block_position", "frame"), ("block_orientation", "frame"),
    ("agent_velocity", "frame"), ("agent_velocity", "delta"),
    ("agent_acceleration", "frame"), ("agent_acceleration", "second_delta"),
    ("block_velocity", "frame"), ("block_velocity", "delta"),
    ("block_acceleration", "frame"), ("block_acceleration", "second_delta"),
    ("block_angular_velocity", "frame"), ("block_angular_velocity", "delta"),
    ("block_angular_acceleration", "frame"), ("block_angular_acceleration", "second_delta"),
]
CARTESIAN_SPECS_PREDICTOR = [
    (variable, "frame") for variable in (
        "agent_position", "agent_velocity", "agent_acceleration",
        "block_position", "block_velocity", "block_acceleration",
        "block_orientation", "block_angular_velocity", "block_angular_acceleration",
    )
]

def read_rows(path):
    def parse(value):
        if value == "":
            return value
        try:
            return int(value)
        except ValueError:
            try:
                return float(value)
            except ValueError:
                return value
    with Path(path).open(newline="") as handle:
        return [
            {key: parse(value) for key, value in row.items()}
            for row in csv.DictReader(handle)
        ]

def primary_mode(family, variable):
    if family == "predictor" or variable in ("agent_position", "block_position", "block_orientation"):
        return "frame"
    if variable.endswith("velocity"):
        return "delta"
    if variable.endswith("acceleration"):
        return "second_delta"
    raise KeyError(variable)

def evaluate_representation(condition, name, rep, variable, mode, split_name, train_idx, test_idx, design_cache):
    features, labels, pose_context = align_representation(rep, targets, variable, mode)
    cache_prefix = (name, mode, split_name)
    design = get_design(design_cache, cache_prefix, features, labels, train_idx, test_idx, RIDGE)
    truth, prediction, _ = design.fit(labels, train_idx, test_idx)
    scores = regression_scores(truth, prediction)
    truth_groups, prediction_groups = group_flat_predictions_by_window(
        features, labels, test_idx, truth, prediction
    )
    ci = trajectory_bootstrap_metric_ci(
        truth_groups, prediction_groups, "r2", BOOTSTRAP_REPEATS, SEED
    )
    shuffled = vectorized_shuffled_label_scores(
        design, labels, train_idx, test_idx, SHUFFLE_REPEATS, SEED
    )
    pose_design = RidgeDesign.prepare(pose_context, labels, train_idx, test_idx, RIDGE)
    pose_truth, pose_prediction, _ = pose_design.fit(labels, train_idx, test_idx)
    residual_labels = residualize_against_context(labels, pose_context, train_idx, RIDGE)
    residual_design = get_design(
        design_cache, cache_prefix + ("residual",), features, residual_labels,
        train_idx, test_idx, RIDGE,
    )
    residual_truth, residual_prediction, _ = residual_design.fit(
        residual_labels, train_idx, test_idx
    )
    family, layer, kind = name.split("/", 2)
    return {
        "condition": condition, "representation": name, "family": family,
        "layer": int(layer), "kind": kind, "variable": variable, "mode": mode,
        "split": split_name, **scores, "ci_low": ci[0], "ci_high": ci[1],
        "shuffled_q95": float(np.quantile(shuffled, 0.95)),
        "pose_only_r2": regression_scores(pose_truth, pose_prediction)["r2"],
        "pose_residual_r2": regression_scores(residual_truth, residual_prediction)["r2"],
    }

def evaluate_condition(condition, condition_representations):
    rows = []
    total = len(condition_representations) * len(splits)
    done = 0
    for split_name, (train_idx, test_idx) in splits.items():
        for name, rep in sorted(condition_representations.items()):
            family = name.split("/", 1)[0]
            specs = CARTESIAN_SPECS_DINO if family == "dino" else CARTESIAN_SPECS_PREDICTOR
            design_cache = {}
            for variable, mode in specs:
                rows.append(evaluate_representation(
                    condition, name, rep, variable, mode, split_name,
                    train_idx, test_idx, design_cache,
                ))
            done += 1
            if done % 5 == 0 or done == total:
                print(f"{condition} probe representations {done}/{total}")
            del design_cache
            gc.collect()
    return rows

off_metrics_path = OUTPUT_DIR / "off_layerwise_cartesian_metrics.csv"
reuse_off_metrics = False
if off_metrics_path.exists():
    off_metrics = read_rows(off_metrics_path)
    reuse_off_metrics = {row.get("split") for row in off_metrics} == set(splits)
if reuse_off_metrics:
    print(f"Reusing {len(off_metrics):,} development-only OFF Cartesian probe rows")
else:
    off_metrics = evaluate_condition("off", representations)
    write_rows(off_metrics_path, off_metrics)
assert off_metrics and {row["condition"] for row in off_metrics} == {"off"}
show_rows(off_metrics, limit=12)

## 5. Where each Cartesian and circular variable becomes readable

DINO and predictor curves are deliberately separated. DINO uses blocks 0 through 11. Predictor uses blocks 0 through 5. Each figure shows only one representation family and one physical body.

In [ ]:
TARGET_GROUPS = {
    "pusher": ["agent_position", "agent_velocity", "agent_acceleration"],
    "block_translation": ["block_position", "block_velocity", "block_acceleration"],
    "block_rotation": ["block_orientation", "block_angular_velocity", "block_angular_acceleration"],
}

def plot_family_group(rows, family, group_name, split_name, prefix, condition=None):
    variables = TARGET_GROUPS[group_name]
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.7), constrained_layout=True)
    for ax, variable in zip(axes, variables):
        candidates = [
            row for row in rows
            if row["family"] == family and row["split"] == split_name
            and row["variable"] == variable
            and row["mode"] == primary_mode(family, variable)
            and (condition is None or row.get("condition") == condition)
        ]
        for (kind,), group in grouped(candidates, ["kind"]).items():
            group = sorted(group, key=lambda row: row["layer"])
            layer = np.asarray([row["layer"] for row in group])
            ax.plot(layer, [row["r2"] for row in group], marker="o", label=kind)
            ax.fill_between(layer, [row["ci_low"] for row in group], [row["ci_high"] for row in group], alpha=0.12)
        ax.axhline(0, color="black", lw=1)
        ax.set(title=variable.replace("_", " ").title(), xlabel=f"{family.title()} block index", ylabel="Held-out R²")
        ax.legend(fontsize=7)
    filename = f"{prefix}_{family}_{group_name}_{split_name}.png"
    fig.savefig(OUTPUT_DIR / filename, dpi=180)
    plt.show()

for family in ("dino", "predictor"):
    for group_name in TARGET_GROUPS:
        for split_name in splits:
            plot_family_group(off_metrics, family, group_name, split_name, "off", "off")

## 6. Is DINO motion static or genuinely temporal?

Raw-frame DINO motion is a shortcut diagnostic because DINO sees one image at a time. The stronger tests use first differences for velocity and second differences for acceleration. A temporal score that survives the spatial-validation split and pose controls is harder to explain using location alone.

In [ ]:
def plot_static_temporal(rows, group_name, filename):
    variables = TARGET_GROUPS[group_name][1:]
    fig, axes = plt.subplots(2, 2, figsize=(15, 9), constrained_layout=True)
    for ax, split_name, variable in zip(
        axes.flat,
        ["episode_validation", "episode_validation", "spatial_validation", "spatial_validation"],
        variables + variables,
    ):
        candidates = [
            row for row in rows if row["family"] == "dino" and row["split"] == split_name
            and row["variable"] == variable
        ]
        for (mode, kind), group in grouped(candidates, ["mode", "kind"]).items():
            group = sorted(group, key=lambda row: row["layer"])
            ax.plot([row["layer"] for row in group], [row["r2"] for row in group], marker="o", label=f"{mode}: {kind}")
        ax.axhline(0, color="black", lw=1)
        ax.set(title=f"{split_name.replace('_', ' ')}: {variable.replace('_', ' ')}", xlabel="DINO block", ylabel="Held-out R²")
        ax.legend(fontsize=6)
    fig.savefig(OUTPUT_DIR / filename, dpi=180)
    plt.show()

plot_static_temporal(off_metrics, "pusher", "off_static_vs_temporal_dino_pusher.png")
plot_static_temporal(off_metrics, "block_translation", "off_static_vs_temporal_dino_block_translation.png")
plot_static_temporal(off_metrics, "block_rotation", "off_static_vs_temporal_dino_block_rotation.png")

## 7. Cartesian versus polar and circular motion

Translational probes compare Cartesian vectors with magnitude and direction. Direction uses mean cosine similarity and excludes the slowest ten percent of training samples because heading is unstable near zero speed. T-block orientation already uses a circular cosine-sine target in the main metrics.

In [ ]:
POLAR_SPECS = [
    ("agent_speed", "agent_heading", "agent_velocity"),
    ("agent_acceleration_magnitude", "agent_acceleration_direction", "agent_acceleration"),
    ("block_speed", "block_heading", "block_velocity"),
    ("block_acceleration_magnitude", "block_acceleration_direction", "block_acceleration"),
]

def evaluate_polar_condition(condition, condition_representations):
    rows = []
    train_idx, test_idx = splits["episode_validation"]
    for name, rep in sorted(condition_representations.items()):
        family, layer, kind = name.split("/", 2)
        for magnitude_variable, direction_variable, base_variable in POLAR_SPECS:
            mode = primary_mode(family, base_variable)
            features, magnitude, _ = align_representation(rep, targets, magnitude_variable, mode)
            _, direction, _ = align_representation(rep, targets, direction_variable, mode)
            direction, cutoff = mask_slow_directions(direction, magnitude, train_idx, 0.1)

            magnitude_design = RidgeDesign.prepare(features, magnitude, train_idx, test_idx, RIDGE)
            truth, prediction, _ = magnitude_design.fit(magnitude, train_idx, test_idx)
            score = regression_scores(truth, prediction)
            truth_groups, prediction_groups = group_flat_predictions_by_window(
                features, magnitude, test_idx, truth, prediction
            )
            ci = trajectory_bootstrap_metric_ci(
                truth_groups, prediction_groups, "r2", BOOTSTRAP_REPEATS, SEED
            )
            rows.append({
                "condition": condition, "representation": name, "family": family,
                "layer": int(layer), "kind": kind, "variable": magnitude_variable,
                "mode": mode, "split": "episode_validation", "protocol": "train_val_test_v2",
                **score, "ci_low": ci[0], "ci_high": ci[1],
            })

            direction_design = RidgeDesign.prepare(features, direction, train_idx, test_idx, RIDGE)
            truth, prediction, _ = direction_design.fit(direction, train_idx, test_idx)
            score = direction_scores(truth, prediction)
            truth_groups, prediction_groups = group_flat_predictions_by_window(
                features, direction, test_idx, truth, prediction
            )
            ci = trajectory_bootstrap_metric_ci(
                truth_groups, prediction_groups, "cosine", BOOTSTRAP_REPEATS, SEED
            )
            rows.append({
                "condition": condition, "representation": name, "family": family,
                "layer": int(layer), "kind": kind, "variable": direction_variable,
                "mode": mode, "split": "episode_validation", "protocol": "train_val_test_v2",
                **score, "ci_low": ci[0], "ci_high": ci[1],
                "slow_cutoff": cutoff,
            })
        gc.collect()
    return rows

off_polar_path = OUTPUT_DIR / "off_layerwise_polar_metrics.csv"
reuse_off_polar = False
if off_polar_path.exists():
    off_polar_metrics = read_rows(off_polar_path)
    reuse_off_polar = {row.get("protocol") for row in off_polar_metrics} == {"train_val_test_v2"}
if reuse_off_polar:
    print(f"Reusing {len(off_polar_metrics):,} development-only OFF polar probe rows")
else:
    off_polar_metrics = evaluate_polar_condition("off", representations)
    write_rows(off_polar_path, off_polar_metrics)
assert off_polar_metrics and {row["condition"] for row in off_polar_metrics} == {"off"}

def plot_polar(rows, family, body, prefix, condition=None):
    variables = (
        ["agent_speed", "agent_heading", "agent_acceleration_magnitude", "agent_acceleration_direction"]
        if body == "pusher" else
        ["block_speed", "block_heading", "block_acceleration_magnitude", "block_acceleration_direction"]
    )
    fig, axes = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)
    for ax, variable in zip(axes.flat, variables):
        candidates = [
            row for row in rows if row["family"] == family and row["variable"] == variable
            and (condition is None or row.get("condition") == condition)
        ]
        score_key = "cosine" if variable.endswith("heading") or variable.endswith("direction") else "r2"
        for (kind,), group in grouped(candidates, ["kind"]).items():
            group = sorted(group, key=lambda row: row["layer"])
            ax.plot([row["layer"] for row in group], [row[score_key] for row in group], marker="o", label=kind)
        ax.axhline(0, color="black", lw=1)
        ax.set(title=variable.replace("_", " ").title(), xlabel=f"{family.title()} block", ylabel="R²" if score_key == "r2" else "Mean cosine similarity")
        ax.legend(fontsize=7)
    fig.savefig(OUTPUT_DIR / f"{prefix}_{family}_{body}_polar.png", dpi=180)
    plt.show()

for family in ("dino", "predictor"):
    for body in ("pusher", "block"):
        plot_polar(off_polar_metrics, family, body, "off", "off")

## 8. Emergence table

A conservative onset is the first of two consecutive layers whose held-out score exceeds the shuffled-label 95th percentile and reaches at least half of that representation family's peak score.

In [ ]:
off_onsets = []
primary_rows = [
    row for row in off_metrics if row["split"] == "episode_validation"
    and row["mode"] == primary_mode(row["family"], row["variable"])
]
for (family, kind, variable), group in grouped(primary_rows, ["family", "kind", "variable"]).items():
    best = max(group, key=lambda row: row["r2"])
    off_onsets.append({
        "condition": "off", "family": family, "kind": kind, "variable": variable,
        "onset_layer": readability_onset(group, "r2", "shuffled_q95", 2, 0.5),
        "peak_r2": best["r2"], "peak_layer": int(best["layer"]),
    })
write_rows(OUTPUT_DIR / "off_readability_onsets.csv", off_onsets)
show_rows(sorted(off_onsets, key=lambda row: (row["variable"], row["family"], row["kind"])), limit=40)

## 9. Interpretation checklist

Read the outputs in this order:

1. Inspect the dataset overview and spatial split.
2. Read the DINO pusher, block-translation, and block-rotation figures separately.
3. Read the predictor figures separately. Predictor block 0 is not DINO layer 12.
4. Compare raw-frame and temporal-difference DINO curves.
5. Check shuffled, pose-only, pose-residualized, and spatial-holdout values before calling a variable readable.
6. Treat block orientation as circular. Do not interpret raw wrapped-angle distance linearly.
7. Do not infer causal planner use from a probe result.

In [ ]:
initial_summary = {
    "run_label": RUN_LABEL,
    "trajectory_limit": TRAJECTORY_LIMIT,
    "off_cache": cache_metadata,
    "ridge": RIDGE,
    "bootstrap_repeats": BOOTSTRAP_REPEATS,
    "shuffle_repeats": SHUFFLE_REPEATS,
    "episode_split": {"train": len(episode_train), "validation": len(episode_validation), "locked_test": len(episode_test)},
    "spatial_split": spatial_config,
    "off_onsets": off_onsets,
    "off_best_rows": sorted(off_metrics, key=lambda row: row["r2"], reverse=True)[:40],
}
(OUTPUT_DIR / "off_summary.json").write_text(json.dumps(initial_summary, indent=2, default=str))
print("OFF outputs:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(" -", path.name)

## 10. Recovered straightening comparison across 18,500 trajectories

This section keeps the exact OFF sample fixed and collects ON activations for those same frames. The architecture, seed, dataset, two-epoch schedule, window identifiers, labels, splits, representations, and probe settings match.

The recovered configs use encoder learning rate `1e-6` for OFF and `1e-5` for ON. Per request, the notebook does not block on that difference. The output is therefore a comparison of the two recovered trained systems, not an estimate that isolates straightening alone.

In [ ]:
from omegaconf import OmegaConf
configs = {condition: OmegaConf.load(path) for condition, path in CONFIGS.items()}
checkpoint_provenance = {
    condition: {
        "checkpoint": str(CHECKPOINTS[condition]),
        "checkpoint_sha256": CHECKPOINT_SHA256[condition],
        "epoch": int(configs[condition].training.epochs),
        "seed": int(configs[condition].training.seed),
        "straighten": configs[condition].training.straighten,
        "encoder_lr": float(configs[condition].training.encoder_lr),
        "frameskip": int(configs[condition].frameskip),
        "num_hist": int(configs[condition].num_hist),
        "num_pred": int(configs[condition].num_pred),
    }
    for condition in ("off", "on")
}
assert checkpoint_provenance["off"]["epoch"] == checkpoint_provenance["on"]["epoch"] == 2
assert checkpoint_provenance["off"]["seed"] == checkpoint_provenance["on"]["seed"] == SEED
assert checkpoint_provenance["off"]["frameskip"] == checkpoint_provenance["on"]["frameskip"] == FRAME_SKIP
assert checkpoint_provenance["off"]["num_hist"] == checkpoint_provenance["on"]["num_hist"] == 3
assert len(choices) == TRAJECTORY_LIMIT
print(checkpoint_provenance)

### Reuse the same 18,500 windows and collect ON activations

No trajectories or start frames are resampled. The ON cache must reproduce the OFF states and normalized action blocks exactly before any comparison runs.

In [ ]:
matched_cartesian_path = OUTPUT_DIR / "matched_layerwise_cartesian_metrics.csv"
matched_polar_path = OUTPUT_DIR / "matched_layerwise_polar_metrics.csv"
reuse_on_tables = False
if matched_cartesian_path.exists() and matched_polar_path.exists():
    cached_cartesian = read_rows(matched_cartesian_path)
    cached_polar = read_rows(matched_polar_path)
    reuse_on_tables = (
        {row.get("split") for row in cached_cartesian} == set(splits)
        and {row.get("protocol") for row in cached_polar} == {"train_val_test_v2"}
    )
if reuse_on_tables:
    print("Reusing completed matched OFF/ON probe tables; ON activations remain unchanged")
else:
    on_cache = OUTPUT_DIR / "activation_cache_on"
    on_cache_valid = False
    if (on_cache / "manifest.json").exists():
        on_representations, on_states, on_actions, on_choices, on_cache_metadata = load_activation_cache(on_cache)
        on_cache_valid = np.array_equal(on_choices, choices)
        on_cache_valid = on_cache_valid and on_cache_metadata.get("checkpoint_sha256") == CHECKPOINT_SHA256["on"]
        if not on_cache_valid:
            print("Ignoring the stale ON cache because its windows do not match")

    if not on_cache_valid:
        if not FORCE_RECOMPUTE:
            raise RuntimeError("The shared ON cache failed provenance or window validation; delete it and rerun the cache-download cell.")
        modules, epoch = load_checkpoint_modules(CHECKPOINTS["on"], DEVICE)
        on_representations, on_states, on_actions = collect_activations(modules, dataset, choices)
        on_cache_metadata = {
            "condition": "on", "checkpoint_sha256": CHECKPOINT_SHA256["on"],
            "epoch": epoch, "straighten": "aggcos1e-1", "seed": SEED,
            "num_windows": len(choices), "num_frames": NUM_FRAMES,
            "frameskip": FRAME_SKIP, "ridge": RIDGE,
        }
        save_activation_cache(on_cache, on_representations, on_states, on_actions, choices, on_cache_metadata)
        del modules, on_representations, on_states, on_actions
        gc.collect(); torch.cuda.empty_cache()
        on_representations, on_states, on_actions, on_choices, on_cache_metadata = load_activation_cache(on_cache)

    assert np.array_equal(on_choices, choices)
    assert set(on_representations) == set(representations)
    assert np.array_equal(np.asarray(on_states), np.asarray(states))
    assert np.array_equal(np.asarray(on_actions), np.asarray(actions))
    assert len(on_representations) == 42
    print({
        "windows": len(on_choices), "unique_episodes": len(np.unique(on_choices[:, 0])),
        "off_representations": len(representations), "on_representations": len(on_representations),
    })

### Run the same layerwise probes

ON receives the same targets, episode-validation split, spatial-validation split, ridge value, 20 shuffled-label controls, pose-only control, pose residualization, and complete-window bootstrap procedure used for OFF.

In [ ]:
if reuse_on_tables:
    matched_metrics = read_rows(matched_cartesian_path)
    matched_polar_metrics = read_rows(matched_polar_path)
    on_metrics = [row for row in matched_metrics if row["condition"] == "on"]
    on_polar_metrics = [row for row in matched_polar_metrics if row["condition"] == "on"]
    assert {row["condition"] for row in matched_metrics} == {"off", "on"}
    assert {row["condition"] for row in matched_polar_metrics} == {"off", "on"}
    print(f"Reusing {len(on_metrics):,} ON Cartesian and {len(on_polar_metrics):,} ON polar rows")
else:
    on_metrics = evaluate_condition("on", on_representations)
    on_polar_metrics = evaluate_polar_condition("on", on_representations)
    matched_metrics = off_metrics + on_metrics
    matched_polar_metrics = off_polar_metrics + on_polar_metrics
    write_rows(matched_cartesian_path, matched_metrics)
    write_rows(matched_polar_path, matched_polar_metrics)
show_rows(on_metrics, limit=12)

### Overlay straightening OFF and ON

Dashed lines are OFF and solid lines are ON. Each output keeps DINO and predictor separate, and also separates pusher, block translation, and block rotation.

In [ ]:
COLORS = {
    "cls": "#4C78A8", "pooled_patches": "#59A14F",
    "projected_aggregate": "#E15759", "pooled_visual": "#B279A2",
}
STYLES = {"off": "--", "on": "-"}

def plot_matched_family_group(family, group_name, split_name):
    variables = TARGET_GROUPS[group_name]
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.7), constrained_layout=True)
    for ax, variable in zip(axes, variables):
        candidates = [
            row for row in matched_metrics if row["family"] == family
            and row["split"] == split_name and row["variable"] == variable
            and row["mode"] == primary_mode(family, variable)
        ]
        for condition in ("off", "on"):
            for (kind,), group in grouped(
                [row for row in candidates if row["condition"] == condition], ["kind"]
            ).items():
                group = sorted(group, key=lambda row: row["layer"])
                layer = np.asarray([row["layer"] for row in group])
                ax.plot(
                    layer, [row["r2"] for row in group], marker="o",
                    color=COLORS[kind], linestyle=STYLES[condition],
                    label=f"{condition.upper()} | {kind}",
                )
                ax.fill_between(layer, [row["ci_low"] for row in group], [row["ci_high"] for row in group], color=COLORS[kind], alpha=0.06)
        ax.axhline(0, color="black", lw=1)
        ax.set(title=variable.replace("_", " ").title(), xlabel=f"{family.title()} block", ylabel="Held-out R²")
        ax.legend(fontsize=6, ncol=2)
    fig.savefig(OUTPUT_DIR / f"matched_{family}_{group_name}_{split_name}.png", dpi=180)
    plt.show()

for family in ("dino", "predictor"):
    for group_name in TARGET_GROUPS:
        for split_name in splits:
            plot_matched_family_group(family, group_name, split_name)

for condition in ("on",):
    for family in ("dino", "predictor"):
        for body in ("pusher", "block"):
            plot_polar(matched_polar_metrics, family, body, "matched_on", condition)

### Read the comparison

A positive ON minus OFF value means the recovered ON model made that target more linearly readable at that layer. Check pose controls, residual scores, shuffled nulls, and the spatial-validation split before interpreting a gap as motion information.

Because the recovered runs use different encoder learning rates, the gap describes the two trained systems and should not be attributed to straightening alone.

In [ ]:
paired_deltas = []
off_lookup = {
    (row["representation"], row["variable"], row["mode"], row["split"]): row
    for row in off_metrics
}
for on_row in on_metrics:
    key = (on_row["representation"], on_row["variable"], on_row["mode"], on_row["split"])
    off_row = off_lookup[key]
    paired_deltas.append({
        "split": on_row["split"], "family": on_row["family"],
        "variable": on_row["variable"], "mode": on_row["mode"],
        "representation": on_row["representation"],
        "off_r2": off_row["r2"], "on_r2": on_row["r2"],
        "on_minus_off_r2": on_row["r2"] - off_row["r2"],
        "off_pose_residual_r2": off_row["pose_residual_r2"],
        "on_pose_residual_r2": on_row["pose_residual_r2"],
    })
write_rows(OUTPUT_DIR / "straightening_on_minus_off.csv", paired_deltas)

final_summary = {
    "status": "complete",
    "run_label": RUN_LABEL,
    "trajectory_limit": TRAJECTORY_LIMIT,
    "unique_episodes": int(len(np.unique(choices[:, 0]))),
    "checkpoint_provenance": checkpoint_provenance,
    "comparison_scope": "recovered runs; encoder learning rates differ",
    "splits": {
        name: {"train": len(train), "validation": len(test)}
        for name, (train, test) in splits.items()
    },
    "largest_absolute_deltas": sorted(
        paired_deltas, key=lambda row: abs(row["on_minus_off_r2"]), reverse=True
    )[:50],
}
(OUTPUT_DIR / "summary.json").write_text(json.dumps(final_summary, indent=2, default=str))

package_dir = Path("/content") / f"{RUN_LABEL}_graphs_and_tables"
if package_dir.exists():
    shutil.rmtree(package_dir)
package_dir.mkdir()
for path in OUTPUT_DIR.iterdir():
    if path.is_file() and path.suffix.lower() in {".png", ".csv", ".json", ".txt", ".yaml"}:
        shutil.copy2(path, package_dir / path.name)
archive = shutil.make_archive(str(OUTPUT_DIR / f"{RUN_LABEL}_graphs_and_tables"), "zip", package_dir)

expected = [
    "dataset_motion_overview.png", "spatial_split.png",
    "off_layerwise_cartesian_metrics.csv", "off_layerwise_polar_metrics.csv",
    "matched_layerwise_cartesian_metrics.csv", "matched_layerwise_polar_metrics.csv",
    "straightening_on_minus_off.csv", "summary.json",
]
missing = [name for name in expected if not (OUTPUT_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f"Missing required outputs: {missing}")
print("PUSHT_PROBE_RUN_COMPLETE")
print({"output_dir": str(OUTPUT_DIR), "archive": archive, "files": len(list(OUTPUT_DIR.iterdir()))})
show_rows(
    sorted(paired_deltas, key=lambda row: abs(row["on_minus_off_r2"]), reverse=True),
    ["split", "family", "variable", "representation", "off_r2", "on_r2", "on_minus_off_r2"],
    limit=30,
)
print("Run the confirmatory section below before downloading the final archive.")


## 11. Confirmatory trajectory-level evaluation

Layer and readout selection is performed only on validation trajectories. For each physical target and model family, one shared layer/readout/temporal construction is chosen by mean OFF/ON validation $R^2$. The same choice is fixed for both conditions before each ridge probe is refit on training plus validation trajectories and evaluated once on locked test trajectories.

All headline $R^2$, RMSE, MAE, and ON-minus-OFF values include 95% percentile intervals from 1,000 bootstrap resamples of complete PushT trajectory windows. The present assets contain model-training seed 0 only; additional model seeds require independently trained OFF and ON checkpoint/cache pairs.

In [ ]:
HEADLINE_BOOTSTRAP_REPEATS = 1000
AVAILABLE_MODEL_SEEDS = [0]

headline_on_cache = OUTPUT_DIR / "activation_cache_on"
if "on_representations" not in globals():
    on_representations, on_states, on_actions, on_choices, on_cache_metadata = load_activation_cache(headline_on_cache)
    assert np.array_equal(on_choices, choices)

def pusht_headline_specs(family, rep):
    return CARTESIAN_SPECS_DINO if family == "dino" else CARTESIAN_SPECS_PREDICTOR

headline_representations = {"off": representations, "on": on_representations}
validation_selection, headline_metrics, headline_deltas = select_then_test_representations(
    headline_representations, targets, align_representation, pusht_headline_specs,
    episode_train, episode_validation, episode_test, ridge=RIDGE,
    bootstrap_repeats=HEADLINE_BOOTSTRAP_REPEATS, seed=SEED, model_seed=0,
)
write_rows(OUTPUT_DIR / "validation_selection_scores.csv", validation_selection)
write_rows(OUTPUT_DIR / "headline_selected_test_metrics.csv", headline_metrics)
write_rows(OUTPUT_DIR / "headline_straightening_deltas.csv", headline_deltas)
headline_protocol = {
    "model_training_seeds": AVAILABLE_MODEL_SEEDS,
    "split_windows": {"train": len(episode_train), "validation": len(episode_validation), "test": len(episode_test)},
    "selection": "shared candidate maximizing mean OFF/ON validation R2 within family and target",
    "test_policy": "selected representation evaluated once after refitting on train+validation",
    "bootstrap": {"unit": "complete trajectory window", "repeats": HEADLINE_BOOTSTRAP_REPEATS, "interval": "95% percentile"},
}
(OUTPUT_DIR / "headline_protocol.json").write_text(json.dumps(headline_protocol, indent=2))
show_rows(headline_metrics, ["condition", "family", "variable", "representation", "mode", "r2", "r2_ci_low", "r2_ci_high", "rmse", "rmse_ci_low", "rmse_ci_high", "mae", "mae_ci_low", "mae_ci_high"], limit=30)
show_rows(headline_deltas, limit=30)



condition | family    | variable                   | representation              | mode         | r2                    | r2_ci_low              | r2_ci_high           | rmse                  | rmse_ci_low          | rmse_ci_high         | mae                   | mae_ci_low            | mae_ci_high          
----------+-----------+----------------------------+-----------------------------+--------------+-----------------------+------------------------+----------------------+-----------------------+----------------------+----------------------+-----------------------+-----------------------+----------------------
off       | dino      | agent_acceleration         | dino/8/pooled_patches       | second_delta | 0.5798377193899803    | 0.560960386217398      | 0.5975036065179514   | 0.974243684898055     | 0.9542454579820495   | 0.9952152230883956   | 0.7391043478193923    | 0.7268811813836128    | 0.7515795487150906   
off       | dino      | agent_position             | dino/9/pooled_pat

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

from pathlib import Path
import shutil
from google.colab import files

export_root = Path("/content/notebook_raw_export")
if export_root.exists():
    shutil.rmtree(export_root)
export_root.mkdir()

folders = {"results": Path(OUTPUT_DIR)}
if "MATCHED_OUTPUT_DIR" in globals():
    folders["matched_results"] = Path(MATCHED_OUTPUT_DIR)

for name, source in folders.items():
    if source.exists():
        shutil.copytree(source, export_root / name)

archive = shutil.make_archive(
    "/content/notebook_raw_export",
    "zip",
    export_root,
)
files.download(archive)
